# Comprensión inicial de los datos

## Objetivo

Verificar la integridad del archivo principal y comprender la estructura del conjunto DVM-CAR antes de aplicar transformaciones.

## Preguntas iniciales

1. ¿Qué representa cada fila?
2. ¿Cuántas observaciones y variables existen?
3. ¿Cuáles son los nombres originales de las variables?
4. ¿Qué tipos asigna pandas?
5. ¿La copia local coincide con la fuente documentada?

In [2]:
from pathlib import Path
import hashlib

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError("No se encontró la raíz del proyecto.")


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "dvm_car_v2"
    / "Ad_table (extra).csv"
)

DATA_PATH

WindowsPath('c:/Users/USUARIO/Desktop/DataScience/used-car-price-ml-service/data/raw/dvm_car_v2/Ad_table (extra).csv')

In [3]:
EXPECTED_SHA256 = (
    "0211abfe049d247bf42ad09bff968de6c81e2c810d6b026e4385ff5cbbbac476"
)

with DATA_PATH.open("rb") as file:
    actual_sha256 = hashlib.file_digest(file, "sha256").hexdigest()

print(f"Archivo encontrado : {DATA_PATH.exists()}")
print(f"SHA-256: {actual_sha256}")

assert actual_sha256 == EXPECTED_SHA256, "El archivo  no coincide con la fuente verificada"

Archivo encontrado : True
SHA-256: 0211abfe049d247bf42ad09bff968de6c81e2c810d6b026e4385ff5cbbbac476


In [4]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Filas: {df_raw.shape[0]:,}")
print(f"columnas: {df_raw.shape[1]}")

Filas: 268,255
columnas: 24


In [5]:
df_raw.head(3)

,Maker,Genmodel,Genmodel_ID,Adv_ID,Adv_year,Adv_month,Color,Reg_year,Bodytype,Runned_Miles,...,Engine_power,Annual_Tax,Wheelbase,Height,Width,Length,Average_mpg,Top_speed,Seat_num,Door_num
0,Bentley,Arnage,10_1,10_1$$1,2018,4,Silver,2000.0,Saloon,60000,...,NaN,NaN,3116.0,1515.0,2125.0,5390.0,NaN,NaN,5.0,4.0
1,Bentley,Arnage,10_1,10_1$$2,2018,6,Grey,2002.0,Saloon,44000,...,450.0,315,3116.0,1515.0,2125.0,5390.0,13.7 mpg,179 mph,5.0,4.0
2,Bentley,Arnage,10_1,10_1$$3,2017,11,Blue,2002.0,Saloon,55000,...,400.0,315,3116.0,1515.0,2125.0,5390.0,14.7 mpg,155 mph,5.0,4.0


In [6]:
df_raw.columns.tolist()

['Maker',
 ' Genmodel',
 ' Genmodel_ID',
 'Adv_ID',
 'Adv_year',
 'Adv_month',
 'Color',
 'Reg_year',
 'Bodytype',
 'Runned_Miles',
 'Engin_size',
 'Gearbox',
 'Fuel_type',
 'Price',
 'Engine_power',
 'Annual_Tax',
 'Wheelbase',
 'Height',
 'Width',
 'Length',
 'Average_mpg',
 'Top_speed',
 'Seat_num',
 'Door_num']

In [7]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 268255 entries, 0 to 268254
Data columns (total 24 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Maker         268255 non-null  str    
 1    Genmodel     268255 non-null  str    
 2    Genmodel_ID  268255 non-null  str    
 3   Adv_ID        268255 non-null  str    
 4   Adv_year      268255 non-null  int64  
 5   Adv_month     268255 non-null  int64  
 6   Color         246380 non-null  str    
 7   Reg_year      268248 non-null  float64
 8   Bodytype      267301 non-null  str    
 9   Runned_Miles  267200 non-null  str    
 10  Engin_size    266191 non-null  str    
 11  Gearbox       268088 non-null  str    
 12  Fuel_type     267846 non-null  str    
 13  Price         267110 non-null  float64
 14  Engine_power  236444 non-null  float64
 15  Annual_Tax    221580 non-null  str    
 16  Wheelbase     240257 non-null  float64
 17  Height        240454 non-null  float64
 18  Width         2

## 2. Auditoría estructural

En esta sección se cuantifican valores faltantes, cardinalidad y duplicados sin modificar los datos originales.

In [8]:
column_summary = pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "non_null": df_raw.notna().sum(),
        "missing_pct": df_raw.isna().mean().mul(100).round(3),
        "unique_non_null": df_raw.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)

column_summary

,dtype,non_null,missing_pct,unique_non_null
Annual_Tax,str,221580,17.399,28
Top_speed,str,224653,16.254,138
Average_mpg,str,226322,15.632,358
Engine_power,float64,236444,11.858,466
Width,float64,240175,10.468,475
Wheelbase,float64,240257,10.437,427
Length,float64,240454,10.364,984
Height,float64,240454,10.364,579
Color,str,246380,8.155,22
Seat_num,float64,261781,2.413,10


In [9]:
integrity_summary = pd.Series(
    {
        "rows": len(df_raw),
        "fully_duplicated_rows": df_raw.duplicated().sum(),
        "missing_adv_id": df_raw["Adv_ID"].isna().sum(),
        "duplicated_adv_id": df_raw["Adv_ID"].duplicated().sum(),
    },
    name="value",
)

integrity_summary

rows                     268255
fully_duplicated_rows         0
missing_adv_id                0
duplicated_adv_id             0
Name: value, dtype: int64

In [10]:
header_issues = {
    column: column.strip()
    for column in df_raw.columns
    if column != column.strip()
}

header_issues

{' Genmodel': 'Genmodel', ' Genmodel_ID': 'Genmodel_ID'}

In [11]:
column_summary.reset_index(names="column")

,column,dtype,non_null,missing_pct,unique_non_null
0,Annual_Tax,str,221580,17.399,28
1,Top_speed,str,224653,16.254,138
2,Average_mpg,str,226322,15.632,358
3,Engine_power,float64,236444,11.858,466
4,Width,float64,240175,10.468,475
5,Wheelbase,float64,240257,10.437,427
6,Length,float64,240454,10.364,984
7,Height,float64,240454,10.364,579
8,Color,str,246380,8.155,22
9,Seat_num,float64,261781,2.413,10


In [12]:
numeric_text_columns = [
    "Runned_Miles",
    "Engin_size",
    "Annual_Tax",
    "Average_mpg",
    "Top_speed",
]

for column in numeric_text_columns:
    print(f"\n{column}")
    display(
        df_raw[column]
        .value_counts(dropna=False)
        .head(15)
        .to_frame(name="count")
    )


Runned_Miles


,count
Runned_Miles,
10,2616
100000,1507
80000,1455
60000,1419
70000,1370
50000,1275
90000,1249
65000,1084
75000,1073



Engin_size


,count
Engin_size,
2.0L,59289
1.6L,44301
3.0L,23499
1.2L,22045
1.4L,19349
1.0L,16986
1.5L,14916
2.2L,10471
1.8L,7493



Annual_Tax


,count
Annual_Tax,
NaN,46675
140*,32358
30,23869
140,23734
195,23161
120,18353
20,17747
155,16109
0,13915



Average_mpg


,count
Average_mpg,
NaN,41933
62.8 mpg,8767
55.4 mpg,8189
47.9 mpg,7976
47.1 mpg,7961
54.3 mpg,6949
51.4 mpg,6419
57.6 mpg,6285
65.7 mpg,6214



Top_speed


,count
Top_speed,
NaN,43602
112 mph,15697
155 mph,10841
118 mph,10082
130 mph,9004
99 mph,7629
124 mph,7398
106 mph,7319
109 mph,7058


In [13]:
adv_month_counts = (
    df_raw["Adv_month"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Adv_month")
    .to_frame(name="count")
)

adv_month_counts

,count
Adv_month,
1,6074
2,9851
3,21777
4,41318
5,66999
6,25918
7,35268
8,51067
9,1732


In [14]:
invalid_months_by_year = pd.crosstab(
    df_raw.loc[df_raw["Adv_month"] > 12, "Adv_month"],
    df_raw.loc[df_raw["Adv_month"] > 12, "Adv_year"],
)
invalid_months_by_year

Adv_year,2012
Adv_month,
13,1
17,1
33,1


In [15]:
model_id_consistency = (
    df_raw.groupby(" Genmodel_ID")
    .agg(
        maker_count=("Maker", "nunique"),
        model_count=(" Genmodel", "nunique"),
        advertisement_count=("Adv_ID", "size"),
    )
)

inconsistent_model_ids = model_id_consistency.loc[
    (model_id_consistency["maker_count"] > 1)
    | (model_id_consistency["model_count"] > 1)
]

print(f"Identificadores incosistentes: {len(inconsistent_model_ids)}")
inconsistent_model_ids.head(20)

Identificadores incosistentes: 12


,maker_count,model_count,advertisement_count
Genmodel_ID,,,
21_1,2,3,1612
21_3,2,2,597
21_5,2,2,235
48_1,1,2,96
48_14,1,2,24
48_15,1,3,47
48_18,1,4,534
48_2,1,6,197
48_21,1,4,93


In [16]:
model_name_consistency = (
    df_raw.groupby(["Maker", " Genmodel"])[" Genmodel_ID"]
    .nunique()
    .sort_values(ascending=False)
)

model_name_consistency.head(20)

Maker          Genmodel   
Abarth        124 Spider      1
              595             1
              595C            1
Alfa Romeo    159             1
              4C              1
              Brera           1
              GT              1
              Giulia          1
              Giulietta       1
              MiTo            1
              Spider          1
              Stelvio         1
Aston Martin  DB11            1
              DB9             1
              DBS             1
              Rapide          1
Audi          A1              1
              A2              1
              A3              1
              A3 Cabriolet    1
Name:  Genmodel_ID, dtype: int64

## 3. Diagnóstico de anomalías estructurales

Se examinan los valores que impiden interpretar directamente algunas variables. Esta sección no modifica el conjunto original.

In [17]:
invalid_months_rows = df_raw.loc[
    df_raw["Adv_month"] > 12,
    [
        "Adv_ID",
        "Maker",
        " Genmodel",
        "Adv_year",
        "Adv_month",
        "Reg_year",
        "Runned_Miles",
        "Price",
    ],
]

invalid_months_rows

,Adv_ID,Maker,Genmodel,Adv_year,Adv_month,Reg_year,Runned_Miles,Price
46560,29_32$$75,Ford,Streetka,2012,13,2005.0,22000,2999.0
135473,69_23$$2,Peugeot,407,2012,17,2006.0,77000,4499.0
154764,77_4$$10,Rolls-Royce,Phantom,2012,33,2007.0,20000,116065.0


In [18]:
conflicting_model_details = (
    df_raw.loc[
        df_raw[" Genmodel_ID"].isin(inconsistent_model_ids.index),
        [" Genmodel_ID", "Maker", " Genmodel", "Adv_ID"],
    ]
    .groupby([" Genmodel_ID", "Maker", " Genmodel"])
    .size()
    .rename("advertisement_count")
    .reset_index()
    .sort_values([" Genmodel_ID", "advertisement_count"], ascending=[True, False])
)

conflicting_model_details

,Genmodel_ID,Maker,Genmodel,advertisement_count
0,21_1,Citroen,DS3,1036
2,21_1,DS,3,346
1,21_1,Citroen,DS3 Cabrio,230
3,21_3,Citroen,DS4,460
4,21_3,DS,4,137
5,21_5,Citroen,DS5,157
6,21_5,DS,5,78
8,48_1,Lexus,CT 200h,55
7,48_1,Lexus,CT,41
10,48_14,Lexus,LC 500,23


In [19]:
max_ids_per_model_name = model_name_consistency.max()

models_with_multiple_ids = model_name_consistency[
    model_name_consistency > 1
]

print(f"Mácimo de identificadores por fabricante y modelo: {max_ids_per_model_name}")
print(f"Combinaciones con varios identificadores: {len(models_with_multiple_ids)}")

models_with_multiple_ids.head(20)

Mácimo de identificadores por fabricante y modelo: 1
Combinaciones con varios identificadores: 0


Series([], Name:  Genmodel_ID, dtype: int64)

In [20]:
expected_patterns = {
    "Runned_Miles": r"^\s*\d+(?:\.\d+)?\s*$",
    "Engin_size": r"^\s*\d+(?:\.\d+)?L\s*$",
    "Annual_Tax": r"^\s*\d+(?:\.\d+)?\*?\s*$",
    "Average_mpg": r"^\s*\d+(?:\.\d+)?\s+mpg\s*$",
    "Top_speed": r"^\s*\d+(?:\.\d+)?\s+mph\s*$",
}

format_audit_records = []

for column, pattern in expected_patterns.items():
    values = df_raw[column].dropna().astype(str)
    invalid_values = values.loc[~values.str.match(pattern)]

    format_audit_records.append(
        {
            "column": column,
            "non_null": len(values),
            "invalid_format": len(invalid_values),
            "invalid_format_pct": round(
                len(invalid_values) / len(values) * 100,
                4,
            ),
            "invalid_examples": invalid_values.unique()[:10].tolist(),
        }
    )

format_audit = pd.DataFrame.from_records(format_audit_records)
format_audit

,column,non_null,invalid_format,invalid_format_pct,invalid_examples
0,Runned_Miles,267200,258,0.0966,[1 mile]
1,Engin_size,266191,0,0.0000,[]
2,Annual_Tax,221580,863,0.3895,[*]
3,Average_mpg,226322,0,0.0000,[]
4,Top_speed,224653,0,0.0000,[]


In [21]:
invalid_value_records = []

for column, pattern in expected_patterns.items():
    values = df_raw[column].dropna().astype(str)
    invalid_values = values.loc[~values.str.match(pattern)]

    for raw_value, count in invalid_values.value_counts().items():
        invalid_value_records.append(
            {
                "column": column,
                "raw_value": raw_value,
                "count": count,
            }
        )

invalid_value_sumary = pd.DataFrame.from_records(invalid_value_records)
invalid_value_sumary

,column,raw_value,count
0,Runned_Miles,1 mile,258
1,Annual_Tax,*,863


In [22]:
invalid_values_summary = pd.DataFrame.from_records(
    invalid_value_records
)

invalid_values_summary

,column,raw_value,count
0,Runned_Miles,1 mile,258
1,Annual_Tax,*,863


## 4. Preparación de la copia de trabajo

Las transformaciones se aplican sobre una copia independiente. El objeto `df_raw` se conserva sin modificaciones para mantener la trazabilidad con el archivo original.

### 4.1. Normalización de encabezados

Se eliminan únicamente los espacios situados al inicio o al final de los nombres de las columnas.

In [23]:
df = df_raw.copy()

normalized_columns = df.columns.str.strip()

assert normalized_columns.is_unique, (
    "La normalización produciría nombres de columnas duplicados."
)

df.columns = normalized_columns

print(f"Dimensiones originales: {df_raw.shape}")
print(f"Dimensiones de trabajo: {df.shape}")
print(
    "Encabezados con espacios en df:",
    sum(column != column.strip() for column in df.columns),
)
print(
    "Encabezados con espacios en df_raw:",
    sum(column != column.strip() for column in df_raw.columns),
)

Dimensiones originales: (268255, 24)
Dimensiones de trabajo: (268255, 24)
Encabezados con espacios en df: 0
Encabezados con espacios en df_raw: 2


In [24]:
header_comparison = pd.DataFrame(
    {
        "original_header": df_raw.columns,
        "normalized_header": df.columns,
        "was_modified": df_raw.columns != df.columns,
    }
)

header_comparison.loc[header_comparison["was_modified"]]

,original_header,normalized_header,was_modified
1,Genmodel,Genmodel,True
2,Genmodel_ID,Genmodel_ID,True


### 4.2. Conversión de `Runned_Miles`

La variable contiene valores numéricos almacenados como texto y 258 registros con la unidad `mile`. Se elimina únicamente esta unidad y se valida la conversión antes de actualizar la copia de trabajo.

In [25]:
mileage_text = (
    df["Runned_Miles"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s*miles?$", "", regex= True)
)

mileage_parsed = pd.to_numeric(
    mileage_text,
    errors = "coerce",
)

mileage_parsed.head()

0    60000
1    44000
2    55000
3    14000
4    61652
Name: Runned_Miles, dtype: Int64

In [26]:
mileage_conversion_audit = pd.Series(
    {
        "source_non_null": df["Runned_Miles"].notna().sum(),
        "parsed_non_null": mileage_parsed.notna().sum(),
        "source_missing": df["Runned_Miles"].isna().sum(),
        "parsed_missing": mileage_parsed.isna().sum(),
        "unexpected_conversion_failures": (
            df["Runned_Miles"].notna()
            & mileage_parsed.isna()
        ).sum(),
        "non_integer_values": (
            mileage_parsed.dropna().mod(1).ne(0).sum()
        ),
    },
    name = "value",
)
mileage_conversion_audit

source_non_null                   267200
parsed_non_null                   267200
source_missing                      1055
parsed_missing                      1055
unexpected_conversion_failures         0
non_integer_values                     0
Name: value, dtype: int64

In [27]:
pd.DataFrame(
    {
        "original": df.loc[
            df["Runned_Miles"].eq("1 mile"),
            "Runned_Miles",
        ].head(),
        "parsed": mileage_parsed.loc[
            df["Runned_Miles"].eq("1 mile")
        ].head(),
    }
)

,original,parsed
3173,1 mile,1
3355,1 mile,1
3953,1 mile,1
6389,1 mile,1
9191,1 mile,1


In [28]:
assert mileage_conversion_audit[
    "unexpected_conversion_failures"
] == 0

assert mileage_conversion_audit[
    "non_integer_values" 
] == 0

df["Runned_Miles"] = mileage_parsed.astype("Int64")

print(df["Runned_Miles"].dtype)
print(df["Runned_Miles"].isna().sum())

Int64
1055


In [29]:
print("Tipo en df_raw:", df_raw["Runned_Miles"].dtype)
print("Tipo en df:", df["Runned_Miles"].dtype)

print(
    "Casos '1 mile' en df_raw:",
    df_raw["Runned_Miles"].eq("1 mile").sum(),
)

Tipo en df_raw: str
Tipo en df: Int64
Casos '1 mile' en df_raw: 258


In [30]:
mileage_conversion_audit = pd.Series(
    {
        "source_non_null": df["Runned_Miles"].notna().sum(),
        "parsed_non_null": mileage_parsed.notna().sum(),
        "source_missing": df["Runned_Miles"].isna().sum(),
        "parsed_missing": mileage_parsed.isna().sum(),
        "unexpected_conversion_failures": (
            df["Runned_Miles"].notna()
            & mileage_parsed.isna()
        ).sum(),
        "non_integer_values": (
            mileage_parsed.dropna()
            .mod(1)
            .ne(0)
            .sum()
        ),
    },
    name = "value"
)

mileage_conversion_audit

source_non_null                   267200
parsed_non_null                   267200
source_missing                      1055
parsed_missing                      1055
unexpected_conversion_failures         0
non_integer_values                     0
Name: value, dtype: int64

In [31]:
mileage_special_case = pd.DataFrame(
    {
        "original": df.loc[
            df["Runned_Miles"].eq("1 mile"),
            "Runned_Miles",
        ].head(),
        "parsed": mileage_parsed.loc[
            df["Runned_Miles"].eq("1 mile")
        ].head(),
    }
)

mileage_special_case

,original,parsed


In [32]:
assert mileage_conversion_audit[
    "unexpected_conversion_failures"
] == 0

assert mileage_conversion_audit[
    "non_integer_values"
] == 0

df["Runned_Miles"] = mileage_parsed.astype("Int64")

print("Tipo convertido:", df["Runned_Miles"].dtype)
print("Valores faltantes:", df["Runned_Miles"].isna().sum())

Tipo convertido: Int64
Valores faltantes: 1055


In [33]:
print("Tipo en df_raw:", df_raw["Runned_Miles"].dtype)
print("Tipo en df:", df["Runned_Miles"].dtype)

print(
    "Casos '1 mile' conservados en df_raw:",
    df_raw["Runned_Miles"].eq("1 mile").sum(),
)

Tipo en df_raw: str
Tipo en df: Int64
Casos '1 mile' conservados en df_raw: 258
